# dim_team Backfill Notebook

**Issue #175** — Fix dim_team_backfill to resolve missing teams directly by
numeric team ID from the `game` table.

Detects team IDs referenced in `game.home_team_id` or `game.away_team_id`
that have no matching row in `team.team_id`, then fetches full team detail
from `https://api.nhle.com/stats/rest/en/team/id/{id}` and upserts each
into the `team` table.

## Run instructions

```bash
pip install jupyter pandas httpx sqlalchemy
jupyter notebook nhl-dashboard/notebooks/dim_team_backfill.ipynb
```

Run all cells top-to-bottom.

## Notebook structure

| Section | Content |
|---|---|
| Setup | Imports, SQLAlchemy engine pointing to `instance/nhl.db`, NHL Stats API base URL |
| Section 1 — Detect missing teams | Union `home_team_id`/`away_team_id` from `game`; subtract `team.team_id`; produce `MISSING_TEAM_IDS` |
| Section 2 — Fetch per-team detail | `GET /stats/rest/en/team/id/{id}` for each ID in `MISSING_TEAM_IDS`; build `TEAM_DETAIL_MAP` |
| Section 3 — Upsert function | `upsert_team(session, team_dict)` via `session.merge()` on `tri_code` PK |
| Section 4 — Batch upsert | Loop `TEAM_DETAIL_MAP`, upsert, single commit |
| Section 5 — Verification | Before/after counts, confirm `MISSING_TEAM_IDS` ⊆ `team.team_id`, sample DataFrame |

## Setup

Imports, SQLAlchemy connection to `instance/nhl.db`, and the NHL Stats REST API
base URL. No Flask app context — models are imported directly from
`../backend/models.py`. `httpx` is used for all HTTP calls with 50 ms
rate-limiting between requests.

In [ ]:
import sys
import time
from pathlib import Path

import httpx
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

# Add backend to path so we can import models without Flask app context
BACKEND_DIR = Path("../backend").resolve()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

NHL_STATS_BASE = "https://api.nhle.com/stats/rest/en"
DB_PATH = BACKEND_DIR / "instance" / "nhl.db"

# Connect directly to the Flask app's SQLite DB — no app context needed
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
Session = sessionmaker(bind=engine)

print(f"Database  : {DB_PATH}")
print(f"DB exists : {DB_PATH.exists()}")

## Section 1 — Detect missing teams

Queries the `game` table to collect every distinct integer team ID referenced
in `home_team_id` or `away_team_id`, then subtracts IDs already present in
`team.team_id` to produce `MISSING_TEAM_IDS` — the list of integer IDs to
backfill.

In [ ]:
with engine.connect() as conn:
    # Union both columns so we get every team ID referenced in the game table
    referenced_rows = conn.execute(text(
        "SELECT DISTINCT away_team_id AS team_id FROM game WHERE away_team_id IS NOT NULL "
        "UNION "
        "SELECT DISTINCT home_team_id AS team_id FROM game WHERE home_team_id IS NOT NULL"
    )).fetchall()

    existing_rows = conn.execute(text(
        "SELECT team_id FROM team WHERE team_id IS NOT NULL"
    )).fetchall()

referenced_ids = {row[0] for row in referenced_rows}
existing_ids   = {row[0] for row in existing_rows}
MISSING_TEAM_IDS = sorted(referenced_ids - existing_ids)

print(f"Team IDs referenced in game table : {len(referenced_ids)}")
print(f"Team IDs already in team table    : {len(existing_ids)}")
print(f"Missing team IDs to backfill      : {len(MISSING_TEAM_IDS)}")
print(f"Missing : {MISSING_TEAM_IDS}")

## Section 2 — Fetch per-team detail

For each integer in `MISSING_TEAM_IDS`, calls
`GET https://api.nhle.com/stats/rest/en/team/id/{id}` directly to retrieve
the full team record including `triCode`, `fullName`, `franchiseId`,
`leagueId`, and `rawTricode`.

Stores results in `TEAM_DETAIL_MAP` keyed by integer team ID. Rate-limited
at 50 ms between requests.

In [ ]:
TEAM_DETAIL_MAP = {}  # integer team_id → detail dict from /stats/rest/en/team/id/{id}

for team_id in MISSING_TEAM_IDS:
    try:
        r = httpx.get(f"{NHL_STATS_BASE}/team/id/{team_id}", timeout=15)
        r.raise_for_status()
        detail = r.json()
        # Extract first record from data array
        records = detail.get("data", [])
        if records:
            TEAM_DETAIL_MAP[team_id] = records[0]
            print(f"  OK  id={team_id}: {records[0].get('fullName')} ({records[0].get('triCode')})")
        else:
            print(f"  SKIP id={team_id}: empty data array")
    except Exception as exc:
        print(f"  FAIL id={team_id}: {exc}")
        continue

    time.sleep(0.05)  # polite rate-limiting — 50 ms between requests

print()
print(f"Teams with detail fetched: {len(TEAM_DETAIL_MAP)} of {len(MISSING_TEAM_IDS)} missing")

## Section 3 — Upsert function

Defines `upsert_team(session, team_dict)` which:

1. Extracts all `team` table columns from a per-team detail dict
2. Constructs a `Team` instance with `tri_code`, `team_id`, `franchise_id`,
   `full_name`, `league_id`, `raw_tricode`, and `name` (alias for `fullName`)
3. Calls `session.merge()` to upsert by `tri_code` primary key — idempotent
   on repeated runs

Note: `tri_code` comes from the API response field `triCode`, not from the
`game` table. The `game` table only stores numeric IDs.

In [ ]:
# Import the SQLAlchemy model — keeps column definitions in sync with live schema
from models import Team


def upsert_team(session, team_dict: dict) -> None:
    """Upsert one row into the team table from a /stats/rest/en/team/id/{id} response.

    Uses session.merge() so the call is idempotent: INSERT on first run,
    UPDATE on subsequent runs. tri_code is the primary key.

    Args:
        session: SQLAlchemy Session bound to instance/nhl.db.
        team_dict: Raw team object from /stats/rest/en/team/id/{id} data array.
    """
    full_name  = team_dict.get("fullName")
    raw_tri    = team_dict.get("rawTricode")
    tri_code   = team_dict.get("triCode") or raw_tri

    row = Team(
        tri_code     = tri_code,
        name         = full_name,
        team_id      = team_dict.get("id"),
        franchise_id = team_dict.get("franchiseId"),
        full_name    = full_name,
        league_id    = team_dict.get("leagueId"),
        raw_tricode  = raw_tri,
    )
    session.merge(row)


print("upsert_team() defined")
print("Columns populated:")
for col in ["tri_code", "name", "team_id", "franchise_id", "full_name", "league_id", "raw_tricode"]:
    print(f"  {col}")

## Section 4 — Batch upsert

Loops over all entries in `TEAM_DETAIL_MAP` and calls `upsert_team()` for
each. Commits once after the full batch — the dataset is small (at most a
handful of missing teams), so per-row commits are unnecessary. Individual
failures are logged and skipped without aborting the loop.

In [ ]:
upserted = 0
failed   = 0

with Session() as session:
    for team_id, detail in TEAM_DETAIL_MAP.items():
        try:
            upsert_team(session, detail)
            upserted += 1
            print(f"  OK  id={team_id}: {detail.get('fullName')}")
        except Exception as exc:
            print(f"  FAIL id={team_id}: {exc}")
            failed += 1

    session.commit()  # commit once after the batch

print()
print(f"Batch complete — upserted: {upserted}, failed: {failed}")

## Section 5 — Verification

Queries the `team` table to confirm the backfill succeeded:

1. **Before/after row counts** — compares `COUNT_BEFORE` (captured below)
   with the current count to confirm new rows were inserted
2. **Confirm MISSING_TEAM_IDS resolved** — verifies every integer in
   `MISSING_TEAM_IDS` now appears in `team.team_id`
3. **Sample of newly inserted rows** — displays a DataFrame of the teams
   that were just upserted

In [ ]:
# Capture the before-count (run this cell before Section 4 to get an accurate comparison)
with engine.connect() as conn:
    COUNT_BEFORE = conn.execute(text("SELECT COUNT(*) FROM team WHERE team_id IS NOT NULL")).scalar()

print(f"team row count (team_id IS NOT NULL) before backfill : {COUNT_BEFORE}")

In [ ]:
with engine.connect() as conn:
    count_after = conn.execute(text(
        "SELECT COUNT(*) FROM team WHERE team_id IS NOT NULL"
    )).scalar()

    # Confirm all MISSING_TEAM_IDS are now in team.team_id
    if MISSING_TEAM_IDS:
        placeholders = ",".join(str(i) for i in MISSING_TEAM_IDS)
        resolved_rows = conn.execute(text(
            f"SELECT team_id FROM team WHERE team_id IN ({placeholders})"
        )).fetchall()
        resolved_ids = {row[0] for row in resolved_rows}
        still_missing = sorted(set(MISSING_TEAM_IDS) - resolved_ids)
    else:
        resolved_ids  = set()
        still_missing = []

    # Sample of newly upserted teams
    if MISSING_TEAM_IDS:
        sample_sql = f"""
            SELECT tri_code, name, team_id, franchise_id, full_name, league_id, raw_tricode
            FROM team
            WHERE team_id IN ({placeholders})
            ORDER BY tri_code
        """
        sample_rows = conn.execute(text(sample_sql)).fetchall()
    else:
        sample_rows = []

print(f"team row count before backfill : {COUNT_BEFORE}")
print(f"team row count after  backfill : {count_after}")
print(f"Net new rows inserted          : {count_after - COUNT_BEFORE}")
print()
print(f"MISSING_TEAM_IDS resolved      : {len(resolved_ids)} of {len(MISSING_TEAM_IDS)}")
if still_missing:
    print(f"WARNING — still missing: {still_missing}")
else:
    print("All MISSING_TEAM_IDS now present in team.team_id")
print()

if sample_rows:
    df_sample = pd.DataFrame(
        sample_rows,
        columns=["tri_code", "name", "team_id", "franchise_id", "full_name", "league_id", "raw_tricode"],
    )
    print("Newly upserted teams:")
    display(df_sample)
else:
    print("No missing teams detected — team table is already complete.")